In [1]:
# !wget https://data.ncl.ac.uk/ndownloader/articles/24574753/versions/1 -O ../data/Gutenberg.zip
# !mkdir ../data/Gutenberg
# !unzip ../data/Gutenberg.zip -d ../data/Gutenberg/
# !rm ../data/Gutenberg.zip

In [2]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.train.train import *

In [3]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=27 * 18, output_size=6):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        x = self.fc0(x)
        return x

In [4]:
hyperparams = {
    "num_epochs": 256,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.1},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.002,
    "min_epochs_between_expansions": 16,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 2000,
    "weight_decay": 5e-3,
    "diff_threshold": 0.05
}


In [5]:
train_x = np.load("../data/Gutenberg/train_x.npy")
train_y = np.load("../data/Gutenberg/train_y.npy")
valid_x = np.load("../data/Gutenberg/valid_x.npy")
valid_y = np.load("../data/Gutenberg/valid_y.npy")
test_x = np.load('../data/Gutenberg/test_x.npy')
test_y = np.load('../data/Gutenberg/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 27 * 18)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 27 * 18)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 27 * 18)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = max(1, int(finder_fraction * len(train_dataset)))
finder_dataset = Subset(train_dataset, range(n_finder))

bs = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=bs, shuffle=False)
val_loader = DataLoader(valid_dataset, batch_size=bs, shuffle=False, generator=g)
test_loader = DataLoader(test_dataset, batch_size=bs, shuffle=False, generator=g)

In [6]:
base_model = SimpleFCN()
model = convert_to_scaled_net(base_model, layers_to_convert=[base_model.fc0], device=device)

Converting layer 'fc0' to ScaledNet...


In [7]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key="r0A4z5VJhRpfdOKvs2kuqouv3",
    workspace="shi-van",
    project_name="split_data",
)
experiment.log_parameters(hyperparams)
experiment.set_name("gutenberg origin")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/shi-van/split-data/ab676063d888421c8006c915f2a578f9



In [8]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(model_instance):
    return optim.Adam(
        model_instance.parameters(), 
        lr=hyperparams['learning_rate'], 
        weight_decay=hyperparams['weight_decay']
    )

trained_model = train_ssonn(
    model,
    train_loader,
    val_loader,
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams,
    device,
    experiment,
    True,
)
sparse_model = trained_model

--- Starting SSONN Training ---


Train:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch 1/256 | T-Loss: 1.7680 | V-Loss: 1.7481 | Met: 0.2727 | Params: 2922/2922


Epoch 2/256 | T-Loss: 1.7310 | V-Loss: 1.7293 | Met: 0.2809 | Params: 2922/2922


Epoch 3/256 | T-Loss: 1.7146 | V-Loss: 1.7196 | Met: 0.2871 | Params: 2922/2922


Epoch 4/256 | T-Loss: 1.7059 | V-Loss: 1.7143 | Met: 0.2882 | Params: 2922/2922


... (83 outputs left out)

Epoch 88/256 | T-Loss: 1.5205 | V-Loss: 1.6066 | Met: 0.3461 | Params: 4112679/4115873


Epoch 89/256 | T-Loss: 1.5185 | V-Loss: 1.6034 | Met: 0.3485 | Params: 4112677/4115873


Epoch 90/256 | T-Loss: 1.5149 | V-Loss: 1.6086 | Met: 0.3471 | Params: 4112676/4115873


KeyboardInterrupt: 

In [ ]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : gutenberg origin
COMET INFO:     url                   : https://www.comet.com/shi-van/split-data/437b1b126baa470784225f64693efca0
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     expansion event [3] : (64, 1487)
COMET INFO:     lr [151]            : (0.00025, 0.001)
COMET INFO:     params amount [151] : (2922, 2428799)
COMET INFO:     sparsity [151]      : (0.0, 0.014142413245418206)
COMET INFO:     total_params [151]  : (2922, 2442745)
COMET INFO:     train loss [151]    : (1.363088352436369, 1.768018018792976)
COMET INFO:     val accuracy [151]  : (0.2727333333333333, 0.36293333333333333)
COMET INFO:     val loss [151]      : (1.594628487

In [ ]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [ ]:
accuracy, params

In [ ]:
torch.save(sparse_model, 'gutenberg.pt')